In [ ]:


!apt-get -y install fonts-nanum > /dev/null
!pip -q install xgboost shap > /dev/null

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Add the installed font to the font manager
for fpath in fm.findSystemFonts(fontpaths=None, fontext='ttf'):
    fm.fontManager.addfont(fpath)

# Set the font to NanumGothic (or another Nanum font if preferred)
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

print("Font installed and cache rebuilt.")


# 1 데이터 불러오기 & 전처리

import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

# 업로드한 첫 번째 파일 사용
for name in uploaded.keys():
    file_path = name
    break

df = pd.read_excel(file_path)
df = df.dropna(how='any')

# 목표: 생산량 4톤 이상(=1), 미만(=0)
df['생산성달성여부'] = (df['생산량(T/H)'] > 4).astype(int)

# 날짜 파생
df['생산일자'] = pd.to_datetime(df['생산일자'], errors='coerce')
df['Year']    = df['생산일자'].dt.year
df['Quarter'] = df['생산일자'].dt.quarter
df['Month']   = df['생산일자'].dt.month


# 2 피처/타깃 구성

TARGET = '생산성달성여부'
drop_cols = [TARGET, '생산량(T/H)', '생산일자']  # 누설/원본 날짜 제거(Year/Quarter/Month 유지)
X_raw = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
y = df[TARGET].astype(int)

# 원-핫 인코딩(범주형)
cat_cols = X_raw.select_dtypes(include=['object','category']).columns.tolist()
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)


# 3 Train / Test Split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4 XGBoost 학습 (정확도 기준: 보정 유/무 둘 다 비교)

import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score

# 불균형 보정 없는 기본 모델
xgb_no_weight = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.06, max_depth=5,
    subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, min_child_weight=1.0,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', random_state=42, n_jobs=-1
)

# 불균형 보정 있는 모델 (정확도에 불리 비교용)
pos = y_train.sum()
neg = (y_train==0).sum()
scale_pos_weight = neg / max(pos, 1)
xgb_weighted = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.06, max_depth=5,
    subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, min_child_weight=1.0,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', scale_pos_weight=scale_pos_weight,
    random_state=42, n_jobs=-1
)

# 교차검증으로 Accuracy 비교
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def cv_acc(model):
    gs = GridSearchCV(
        model,
        param_grid={},                # 빠르게 비교하려면 빈 그리드로 CV만
        scoring='accuracy',
        cv=cv, n_jobs=-1
    )
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_score_

modelA, accA = cv_acc(xgb_no_weight)
modelB, accB = cv_acc(xgb_weighted)

best_model = modelA if accA >= accB else modelB
print(f"[CV-Accuracy] No-weight={accA:.4f}  |  Weighted={accB:.4f}")
print("=> 선택 모델:", "무보정" if best_model is modelA else "가중치 보정")

# (선택) 정확도 극대화 GridSearch
param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.03, 0.06, 0.1],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}
grid = GridSearchCV(best_model, param_grid, scoring='accuracy', cv=cv, n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

print(f"[Grid Best CV-Accuracy] {grid.best_score_:.4f}")
print("[Best Params]", grid.best_params_)

# 5 성능 평가 (Test set)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:,1]

acc  = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc  = average_precision_score(y_test, y_proba)

print("\n--- 모델 성능 평가 (Test) ---")
print(f"정확도 (Accuracy): {acc:.4f}")
print(f"정밀도 (Precision): {prec:.4f}")
print(f"재현율 (Recall): {rec:.4f}")
print(f"F1 점수 (F1 Score): {f1:.4f}")
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print("-----------------------------")

# 혼동행렬(색상 지정 안 함)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.6,4.2))
plt.imshow(cm, interpolation='nearest')
plt.title("혼동 행렬")
plt.colorbar()
ticks = np.arange(2)
plt.xticks(ticks, ['0(미달성)','1(달성)'])
plt.yticks(ticks, ['0(미달성)','1(달성)'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i,j],'d'), ha='center', va='center')
plt.ylabel('실제값'); plt.xlabel('예측값')
plt.tight_layout(); plt.show()


#6 변수 중요도(Top-20)

importances = clf.feature_importances_
fi = pd.Series(importances, index=X.columns).sort_values(ascending=False)

top_k = 20
plt.figure(figsize=(7.2,7.2))
plt.barh(range(top_k), fi.head(top_k).values[::-1])
plt.yticks(range(top_k), fi.head(top_k).index[::-1])
plt.title("XGBoost 변수 중요도 (Top 20)")
plt.xlabel("중요도")
plt.tight_layout(); plt.show()

fi.head(25)
